### In this notebook, we will learn the basics of Neural Nets from building out single neuron to MLP (Multi Layer Perceptrons)

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import random

In [2]:
class Value:
    def __init__(self, data, _children =(), _op = '' ):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
            
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))

        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        
        out._backward = _backward
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):  
        return self + other
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return self * other**-1
    
    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        n = self.data 
        out = Value(math.exp(n), (self,))

        def _backward():
            self.grad += out.data * out.grad
        
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0 # Because this is the output node and we will start the back propagation from here.

        for node in reversed(topo):
            node._backward()

In [3]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

In [4]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

In [5]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x

In [6]:
x = [2.0, 3.0, -1.0]
n = Neuron(3)
out = n(x)
print(out)

Value(data=-0.5875023415468179)


In [7]:
x = [2.0, 3.0, -1.0]
n = Layer(3, 3)
out = n(x)
print(out)

[Value(data=-0.9980099558636439), Value(data=0.9998680507682058), Value(data=-0.926954765911004)]


In [8]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=-0.727244205961308)

In [9]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, -1.0]

In [10]:
ypred = [n(x) for x in xs]
ypred

[Value(data=-0.727244205961308),
 Value(data=-0.6822065471004192),
 Value(data=-0.5989763974585863),
 Value(data=-0.748280639578149)]

### Now we can see that the error is high because the input values xs are not near to the target value ys. So we will now implement MSE loss function and try to optimize our Neural Nets using that loss function

In [11]:
[(yout - ygt)**2 for ygt, yout in zip(ys, ypred)]

[Value(data=2.9833725470269092),
 Value(data=0.1009926787058381),
 Value(data=0.16081992979529372),
 Value(data=0.0633626364111857)]

We calculated the loss wrt out ground truth ys and we have squared it so that regardless of the negative or positive sign we always have the loss in positive.